# 139 고정 3전문가 미성형 모델: 시드변경 안정성
A/B 고정 + C=shock6평균+innernormal99%+배럴5std innernormal95%. C 트리시드42/7,11/23,101/37, 세점수평균을 사전고정주후보. 해당 규칙은 이전평가로 발견한개발후보이므로신규 독립검증과구분.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables'
na={};nb={}
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.cluster import KMeans\nmemo={};audit=[]\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for view in ['std24']:\n  X=views[view];imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  full=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy)\n  fullscore=expit(full.decision_function(b))\n  pos=np.flatnonzero(yy==1);neg=np.flatnonzero(normal)\n  k=min(2,len(np.unique(a[pos],axis=0)))\n  modes=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(a[pos]) if k>1 else np.zeros(len(pos),int)\n  experts=[]\n  for mode in np.unique(modes):\n   chosen=np.r_[neg,pos[modes==mode]]\n   scores=[]\n   for C in [.1,1.]:\n    model=SVC(C=C,class_weight='balanced',gamma='scale').fit(a[chosen],yy[chosen])\n    scores.append(expit(model.decision_function(b)))\n   for C in [.1,1.]:\n    model=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a[chosen],yy[chosen])\n    scores.append(model.predict_proba(b)[:,1])\n   experts.append(np.mean(scores,axis=0))\n  expert=np.max(experts,axis=0)\n  out[view+'_mode_max']=expert\n  out[view+'_full']=fullscore\n  out[view+'_geometric']=np.sqrt(expert*fullscore)\n  out[view+'_agreement_min']=np.minimum(expert,fullscore)\n  out[view+'_union_max']=np.maximum(expert,fullscore)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",na);exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.neighbors import NearestNeighbors\nmemo={};audit=[]\ncontext=lab[[v+'_raw' for v in ['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']]].to_numpy(float)\ndef localize(ref,qq,X):\n normal=ref[y[ref]==0]\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[normal])\n xx=imp.transform(X)\n ci=SimpleImputer(strategy='median').fit(context[normal])\n cx=ci.transform(context);cs=RobustScaler().fit(cx[normal]);cx=cs.transform(cx)\n nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cx[normal])\n near=nn.kneighbors(cx[qq],return_distance=False)\n floor=np.maximum(xx[normal].std(axis=0)*.2,1e-6)\n out=[]\n for i,ids in zip(qq,near):\n  peers=normal[ids];peers=peers[peers!=i][:30]\n  out.append((xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor))\n return np.asarray(out)\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for tag,X in [('localphysical12',views['physical12'])]:\n  a=localize(ref,ref,X);b=localize(ref,q,X)\n  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\n   pp=[]\n   for seed in [42,7]:\n    m=cls(n_estimators=120,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\n    pp.append(m.predict_proba(b)[:,1])\n   out[tag+'_'+name]=np.mean(pp,axis=0)\n  for C in [.1,1.]:\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\n   out[tag+'_svm'+str(C)]=expit(m.decision_function(b))\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\n  out[tag+'_lr']=m.predict_proba(b)[:,1]\n  out[tag+'_mean']=np.mean([out[tag+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",nb)
sourceC="from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\nval=lambda n:dense[n].to_numpy(float)\nvs=lambda n:val(n+'__past10_std')\nvd=lambda n:val(n+'__delta1')\nve=lambda n:val(n+'__dev10')\nthermal={\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\n}\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\nmemo={};audit=[]\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for view,X in {'shock6':thermal['shock6']}.items():\n  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal])\n  a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  for tag,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\n   pp=[]\n   for seed in [42,7]:\n    m=cls(n_estimators=100,max_depth=4,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\n    pp.append(m.predict_proba(b)[:,1])\n   out[view+'_'+tag]=np.mean(pp,axis=0)\n  for C in [.1,1.]:\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\n   out[view+'_svm'+str(C)]=expit(m.decision_function(b))\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\n  out[view+'_lr']=m.predict_proba(b)[:,1]\n  out[view+'_mean']=np.mean([out[view+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n"
configs=[('seed42_7',[42,7]),('seed11_23',[11,23]),('seed101_37',[101,37])]
engines=[]
for name,seeds in configs:
 ns={};exec(sourceC.replace('for seed in [42,7]:','for seed in '+str(seeds)+':'),ns);engines.append((name,ns))
lab=na['lab'];y=na['y'];g=na['g'];use=na['use'];barrelstd=engines[0][1]['dense'].Barrel_Temperature_5__past10_std.to_numpy()
records=[];audit=[]
for block in pd.unique(g):
 ref=np.flatnonzero((g!=block)&use);q=np.flatnonzero(g==block)
 oa=na['predict'](ref,q)['std24_geometric'];ob=nb['predict'](ref,q)['localphysical12_mean']
 ia=np.full(len(ref),np.nan);ib=np.full(len(ref),np.nan)
 oc=np.column_stack([ns['predict'](ref,q)['shock6_mean'] for _,ns in engines]);ic=np.full((len(ref),len(engines)),np.nan)
 for held in pd.unique(g[ref]):
  mask=g[ref]==held
  ia[mask]=na['predict'](ref[~mask],ref[mask])['std24_geometric']
  ib[mask]=nb['predict'](ref[~mask],ref[mask])['localphysical12_mean']
  for j,(_,ns) in enumerate(engines):ic[mask,j]=ns['predict'](ref[~mask],ref[mask])['shock6_mean']
 ca=na['threshold'](y[ref],ia,.03);cb=nb['threshold'](y[ref],ib,.03)
 base=(oa>=ca)|(ob>=cb);normal=y[ref]==0
 v=barrelstd[ref][normal&np.isfinite(barrelstd[ref])]
 gatecut=float(np.quantile(v,.95,method='higher'));gate=barrelstd[q]>=gatecut
 configs_scores={name:(ic[:,j],oc[:,j]) for j,(name,_) in enumerate(engines)}
 configs_scores['primary_three_seed_mean']=(ic.mean(axis=1),oc.mean(axis=1))
 for name,(inner,out) in configs_scores.items():
  cut=float(np.quantile(inner[normal],.99,method='higher'))
  new=(out>=cut)&gate;pred=base|new
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=lab.TimeStamp.iloc[i],run_id=block,type=lab.type.iloc[i],actual=int(y[i]),prediction=int(pred[j]),baseline=int(base[j]),new=int(new[j]),score=float(out[j]),threshold=cut,gate_value=float(barrelstd[i]),gate_threshold=gatecut))
 audit.append(dict(block=block,train=len(ref),train_shortshot=int(y[ref].sum()),test=len(q),run_overlap=0,timestamp_overlap=0,unlabeled_training_rows=0))
 print('completed',block,flush=True)
p=pd.DataFrame(records);rows=[]
for n,z in p.groupby('model'):
 z=z[z.type.isin(['정상','미성형'])]
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=n,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(audit),'audit')]:obj.to_csv(t/f'rg3_shortshot_139_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print(p[p.type.eq('미성형')].pivot(index='TimeStamp',columns='model',values='prediction').to_string())


completed rg3_R1


completed rg3_R2


completed rg3_R3


completed rg3_R4


completed rg3_R5


                  model  TP  FP  FN  TN       F1
primary_three_seed_mean   5   9   3 557 0.454545
             seed101_37   5   9   3 557 0.454545
              seed11_23   5   9   3 557 0.454545
               seed42_7   5   9   3 557 0.454545
model                primary_three_seed_mean  seed101_37  seed11_23  seed42_7
TimeStamp                                                                    
2020-10-22 01:09:23                        0           0          0         0
2020-10-22 04:48:39                        1           1          1         1
2020-10-22 05:42:13                        0           0          0         0
2020-10-22 06:12:03                        1           1          1         1
2020-10-22 06:58:22                        1           1          1         1
2020-10-23 02:25:54                        0           0          0         0
2020-10-23 05:47:40                        1           1          1         1
2020-10-23 05:54:52                        1         